# Indian Startup Funding Analysis
### Beginner-level, step-by-step data cleaning with Python & pandas

This notebook cleans the raw startup funding dataset one step at a time.
Each cell does ONE thing, with a comment explaining why.

## Step 1: Load the dataset

In [ ]:
# Import the pandas library and give it a short name 'pd'
import pandas as pd
import numpy as np
import re

In [ ]:
# Load the CSV file into a DataFrame (a table)
# encoding='utf-8-sig' removes a hidden BOM character at the start of the file
df = pd.read_csv('startup_funding.csv', encoding='utf-8-sig')

# Print the shape: (number of rows, number of columns)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
# Look at the first 5 rows to understand the data
df.head()

In [ ]:
# Get a quick summary: column names, data types, non-null counts
df.info()

## Step 2: Check for missing values

In [ ]:
# Count how many missing (empty) values are in each column
df.isnull().sum()

## Step 3: Clean up the column names

The raw column names are messy (spaces, inconsistent capitalization).
We rename them to simple, lowercase, underscore-separated names.

In [ ]:
# Remove extra spaces from column names first
df.columns = df.columns.str.strip()

# See the original column names
print(list(df.columns))

In [ ]:
# Rename columns to clean, simple names
df = df.rename(columns={
    'Sr No': 'sr_no',
    'Date dd/mm/yyyy': 'date',
    'Startup Name': 'startup_name',
    'Industry Vertical': 'industry_vertical',
    'SubVertical': 'sub_vertical',
    'City  Location': 'city',
    'Investors Name': 'investors_name',
    'InvestmentnType': 'investment_type',
    'Amount in USD': 'amount_usd',
    'Remarks': 'remarks'
})

# Check the new column names
df.columns

## Step 4: Drop the 'remarks' column

Most values in this column are missing (empty). A column that is mostly
empty doesn't help our analysis, so we remove it.

In [ ]:
# Check how many values are missing in 'remarks'
missing_count = df['remarks'].isnull().sum()
total_rows = len(df)
print(f"Missing: {missing_count} out of {total_rows} rows")

In [ ]:
# Drop the remarks column since it's mostly empty
df = df.drop(columns=['remarks'])

df.columns

## Step 5: Clean up text columns

Some text values have hidden extra characters (invisible spaces,
stray backslash-n text) and extra spacing. We clean these one column
at a time.

In [ ]:
# Find all columns that contain text (not numbers)
text_columns = df.select_dtypes(include='object').columns
print("Text columns:", list(text_columns))

In [ ]:
# Go through each text column and clean it step by step
for col in text_columns:

    # Step A: make sure everything is treated as text
    df[col] = df[col].astype(str)

    # Step B: remove a hidden character called non-breaking space
    df[col] = df[col].str.replace('\u00a0', '', regex=False)

    # Step C: remove stray literal backslash-n text
    df[col] = df[col].str.replace('\\n', '', regex=False)

    # Step D: remove normal spaces from the start and end
    df[col] = df[col].str.strip()

    # Step E: turn the text "nan" (created by astype(str)) back into
    # a real missing value, so pandas counts it correctly
    df[col] = df[col].replace('nan', np.nan)
    df[col] = df[col].replace('', np.nan)

print("Done cleaning text columns")

In [ ]:
# Check missing values again after cleaning
df.isnull().sum()

## Step 6: Clean the 'amount_usd' column

This column has numbers stored as text, with commas (like "20,00,00,000"),
and some rows say things like "undisclosed" or "unknown" instead of a number.
We convert this column into real numbers, and turn the non-numeric text
into a proper missing value.

In [ ]:
# Let's first look at a few different types of values in this column
df['amount_usd'].dropna().sample(10, random_state=1)

In [ ]:
# This function cleans ONE value at a time.
# We will apply it to every row in the column.

def clean_amount(value):

    # If the value is already missing, keep it missing
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # If the value is text like "undisclosed" or "unknown", treat as missing
    if value.lower() in ['undisclosed', 'unknown', 'n/a', 'nan', '']:
        return np.nan

    # Remove commas (from Indian number formatting, e.g. 20,00,000)
    value = value.replace(',', '')

    # Remove a trailing '+' sign if present (e.g. "14342000+")
    value = value.replace('+', '')

    # Try converting the cleaned text into a number
    try:
        return float(value)
    except ValueError:
        # If it still can't be converted, treat it as missing
        return np.nan

In [ ]:
# Apply the cleaning function to every row in the amount_usd column
df['amount_usd'] = df['amount_usd'].apply(clean_amount)

# Check how many values are missing now
missing_amount = df['amount_usd'].isnull().sum()
print(f"Missing amount_usd values: {missing_amount} out of {len(df)} rows")

In [ ]:
# Look at the cleaned column
df['amount_usd'].describe()

## Step 7: Clean the 'date' column

Dates are stored as text in dd/mm/yyyy format, but a few rows have
typos (extra slashes, dots instead of slashes). We convert this column
into real dates that pandas understands.

In [ ]:
# This function tries to convert ONE date value into a real date

def clean_date(value):

    if pd.isna(value):
        return pd.NaT   # NaT = "Not a Time", the missing value for dates

    value = str(value).strip()

    # Try the normal format first: day/month/year
    try:
        return pd.to_datetime(value, format='%d/%m/%Y')
    except ValueError:
        pass

    # If that failed, replace dots and repeated slashes with a single slash
    fixed_value = re.sub(r'[./]+', '/', value)

    try:
        return pd.to_datetime(fixed_value, format='%d/%m/%Y')
    except ValueError:
        return pd.NaT

In [ ]:
# Apply the date cleaning function to every row
df['date'] = df['date'].apply(clean_date)

# Check how many dates failed to convert
missing_dates = df['date'].isnull().sum()
print(f"Rows with unparseable dates: {missing_dates}")

In [ ]:
# Now that 'date' is a real date, we can pull out the year and month
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

df[['date', 'year', 'month']].head()

## Step 8: Standardize the 'industry_vertical' column

The same industry is often written in many different ways, e.g.
"E-commerce", "Ecommerce", "eCommerce" all mean the same thing.
We create a mapping dictionary to merge the most common variations.

In [ ]:
# See how many different spellings exist right now
print("Unique values before cleaning:", df['industry_vertical'].nunique())

# Look at a sample of e-commerce-like spellings
sample = df['industry_vertical'].dropna().unique()
ecommerce_variants = [v for v in sample if 'comm' in v.lower()]
print(ecommerce_variants[:10])

In [ ]:
# A dictionary mapping messy lowercase versions to one clean name
vertical_map = {
    'e-commerce': 'E-commerce',
    'ecommerce': 'E-commerce',
    'e commerce': 'E-commerce',
    'fintech': 'FinTech',
    'fin-tech': 'FinTech',
    'finance': 'FinTech',
    'edtech': 'EdTech',
    'ed-tech': 'EdTech',
    'education': 'EdTech',
    'healthtech': 'HealthTech',
    'healthcare': 'HealthTech',
    'foodtech': 'FoodTech',
    'food tech': 'FoodTech',
    'logistics': 'Logistics',
    'technology': 'Technology',
    'consumer internet': 'Consumer Internet',
}

def standardize_vertical(value):
    if pd.isna(value):
        return np.nan

    lowercase_value = value.strip().lower()

    # If we have a mapping for this value, use it
    if lowercase_value in vertical_map:
        return vertical_map[lowercase_value]

    # Otherwise, just keep the original text (trimmed)
    return value.strip()

In [ ]:
# Apply the standardization
df['industry_vertical_clean'] = df['industry_vertical'].apply(standardize_vertical)

print("Unique values after cleaning:", df['industry_vertical_clean'].nunique())

In [ ]:
# See the top categories now
df['industry_vertical_clean'].value_counts().head(10)

## Step 9: Save the cleaned dataset

In [ ]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv('clean_startup_funding.csv', index=False)

print("Saved clean_startup_funding.csv")
print(f"Final shape: {df.shape[0]} rows, {df.shape[1]} columns")